In [1]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

# Use the kagglehub client library to attach Kaggle resources like competitions, datasets, and models to your session
# Learn more about kagglehub: https://github.com/Kaggle/kagglehub/blob/main/README.md

import kagglehub
# kagglehub.dataset_download('<owner>/<dataset-slug>')

/kaggle/input/datasets/blastchar/telco-customer-churn/WA_Fn-UseC_-Telco-Customer-Churn.csv


In [2]:
import pandas as pd
import numpy as np
import sqlite3


df = pd.read_csv("/kaggle/input/datasets/blastchar/telco-customer-churn/WA_Fn-UseC_-Telco-Customer-Churn.csv")

df.head()


,customerID,gender,SeniorCitizen,Partner,Dependents,tenure,PhoneService,MultipleLines,InternetService,OnlineSecurity,...,DeviceProtection,TechSupport,StreamingTV,StreamingMovies,Contract,PaperlessBilling,PaymentMethod,MonthlyCharges,TotalCharges,Churn
0,7590-VHVEG,Female,0,Yes,No,1,No,No phone service,DSL,No,...,No,No,No,No,Month-to-month,Yes,Electronic check,29.85,29.85,No
1,5575-GNVDE,Male,0,No,No,34,Yes,No,DSL,Yes,...,Yes,No,No,No,One year,No,Mailed check,56.95,1889.5,No
2,3668-QPYBK,Male,0,No,No,2,Yes,No,DSL,Yes,...,No,No,No,No,Month-to-month,Yes,Mailed check,53.85,108.15,Yes
3,7795-CFOCW,Male,0,No,No,45,No,No phone service,DSL,Yes,...,Yes,Yes,No,No,One year,No,Bank transfer (automatic),42.30,1840.75,No
4,9237-HQITU,Female,0,No,No,2,Yes,No,Fiber optic,No,...,No,No,No,No,Month-to-month,Yes,Electronic check,70.70,151.65,Yes


In [3]:
df.info()


<class 'pandas.core.frame.DataFrame'>
RangeIndex: 7043 entries, 0 to 7042
Data columns (total 21 columns):
 #   Column            Non-Null Count  Dtype  
---  ------            --------------  -----  
 0   customerID        7043 non-null   object 
 1   gender            7043 non-null   object 
 2   SeniorCitizen     7043 non-null   int64  
 3   Partner           7043 non-null   object 
 4   Dependents        7043 non-null   object 
 5   tenure            7043 non-null   int64  
 6   PhoneService      7043 non-null   object 
 7   MultipleLines     7043 non-null   object 
 8   InternetService   7043 non-null   object 
 9   OnlineSecurity    7043 non-null   object 
 10  OnlineBackup      7043 non-null   object 
 11  DeviceProtection  7043 non-null   object 
 12  TechSupport       7043 non-null   object 
 13  StreamingTV       7043 non-null   object 
 14  StreamingMovies   7043 non-null   object 
 15  Contract          7043 non-null   object 
 16  PaperlessBilling  7043 non-null   object 


In [4]:
conn = sqlite3.connect('churn.db')   
df.to_sql('raw_customers', conn, if_exists='replace', index=False)

pd.read_sql("SELECT COUNT(*) AS count FROM raw_customers", conn)  

,count
0,7043


In [5]:
conn.executescript("""
DROP TABLE IF EXISTS customers;
CREATE TABLE customers AS
SELECT customerID, gender, SeniorCitizen, Partner, Dependents, tenure, Churn
FROM raw_customers;
""")

In [6]:
conn.executescript("""
DROP TABLE IF EXISTS services;
create table services AS 
select customerID, PhoneService, MultipleLines, InternetService, OnlineSecurity, DeviceProtection,TechSupport,StreamingTV,StreamingMovies
from raw_customers;
""")

In [7]:
conn.executescript("""
drop table if exists billing;
create table billing as 
SELECT customerID, Contract, PaperlessBilling, PaymentMethod,
       MonthlyCharges, TotalCharges
FROM raw_customers;
""")

In [8]:
pd.read_sql("""
SELECT c.customerID, c.Churn, b.Contract, s.InternetService
FROM customers c
JOIN billing b ON c.customerID = b.customerID
JOIN services s ON c.customerID = s.customerID
LIMIT 5
""", conn)

,customerID,Churn,Contract,InternetService
0,7590-VHVEG,No,Month-to-month,DSL
1,5575-GNVDE,No,One year,DSL
2,3668-QPYBK,Yes,Month-to-month,DSL
3,7795-CFOCW,No,One year,DSL
4,9237-HQITU,Yes,Month-to-month,Fiber optic


In [9]:
pd.read_sql("""
SELECT s.InternetService,
       COUNT(*) AS customers,
       SUM(c.Churn = 'Yes') AS churned,
       ROUND(AVG(c.Churn = 'Yes') * 100, 1) AS churn_rate_pct
FROM customers c
JOIN services s ON c.customerID = s.customerID
GROUP BY s.InternetService
ORDER BY churn_rate_pct DESC
""", conn)

,InternetService,customers,churned,churn_rate_pct
0,Fiber optic,3096,1297,41.9
1,DSL,2421,459,19.0
2,No,1526,113,7.4


In [10]:
pd.read_sql("""
select count(*) as customers , b.PaymentMethod , sum(c.churn="Yes") as churned  , round(avg(c.churn="Yes")*100,1) as churn_rate
from customers c join billing b on c.customerID = b.customerID
group by b.PaymentMethod
""",conn)

,customers,PaymentMethod,churned,churn_rate
0,1544,Bank transfer (automatic),258,16.7
1,1522,Credit card (automatic),232,15.2
2,2365,Electronic check,1071,45.3
3,1612,Mailed check,308,19.1


In [11]:
pd.read_sql("""
select count(*) as customers , round(SUM(b.MonthlyCharges),2) as revenue_loss
from customers c join billing b on c.customerID = b.customerID
where c.churn="Yes"
""",conn)

,customers,revenue_loss
0,1869,139130.85


**Churned customers represent about 139K in lost monthly recurring revenue, roughly 2.1M per year, which sets the budget ceiling for any retention program.”**

In [12]:
pd.read_sql("""
SELECT c.Churn,
       COUNT(*) AS customers,
       ROUND(AVG(b.MonthlyCharges), 2) AS avg_monthly_charge,
       ROUND(SUM(b.MonthlyCharges), 2) AS total_monthly_revenue
FROM customers c
JOIN billing b ON c.customerID = b.customerID
GROUP BY c.Churn
""", conn)

,Churn,customers,avg_monthly_charge,total_monthly_revenue
0,No,5174,61.27,316985.75
1,Yes,1869,74.44,139130.85
